In [ ]:
import os
import pickle
import time
import numpy as np
import pandas as pd
import warnings

from scipy.stats import skew, kurtosis, entropy
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.preprocessing import StandardScaler

from mlquantify.mixture import DyS
from mlquantify.meta import QuaDapt

warnings.filterwarnings("ignore")

# ============================================================
# CONFIGURAÇÕES
# ============================================================

MOSS_PKL = "moss_0_100.pkl"

MOSS_FOLDER = "/var/new_homes/julio/mestrado/mestrado-dyssyn/datasets/moss"
DATASETS_ROOT = "/var/new_homes/julio/mestrado/mestrado-dyssyn/datasets/binary"

BATCH_SIZE = 100
N_PREV = 19
REPEATS = 50
SEED = 42

# ============================================================
# FEATURE EXTRACTION
# ============================================================

FEATURE_ORDER = [
    "mean","var","skew","kurt",
    "q10","q25","q50","q75","q90",
    "entropy"
]

def extract_features(scores):
    scores = np.ravel(scores)

    feats = {
        "mean": np.mean(scores),
        "var": np.var(scores),
        "skew": skew(scores),
        "kurt": kurtosis(scores),
        "q10": np.quantile(scores, 0.1),
        "q25": np.quantile(scores, 0.25),
        "q50": np.quantile(scores, 0.5),
        "q75": np.quantile(scores, 0.75),
        "q90": np.quantile(scores, 0.9),
    }

    hist, _ = np.histogram(scores, bins=20, density=True)
    hist += 1e-12
    feats["entropy"] = entropy(hist)

    return np.array([feats[k] for k in FEATURE_ORDER])

# ============================================================
# APP SAMPLING
# ============================================================

def generate_prevalences(n_prev, repeats):
    prevalences = np.linspace(0.05, 0.95, n_prev)
    return [[1-p, p] for p in prevalences] * repeats

def sample_batch(y, prevalence, batch_size):
    pos_idx = np.where(y == 1)[0]
    neg_idx = np.where(y == 0)[0]

    n_pos = int(batch_size * prevalence[1])
    n_neg = batch_size - n_pos

    pos = np.random.choice(pos_idx, n_pos, replace=True)
    neg = np.random.choice(neg_idx, n_neg, replace=True)

    idx = np.concatenate([pos, neg])
    np.random.shuffle(idx)

    return idx

class APP:
    def __init__(self, batch_size, n_prev, repeats, seed):
        self.batch_size = batch_size
        self.prevalences = generate_prevalences(n_prev, repeats)
        np.random.seed(seed)

    def split(self, X, y):
        for prev in self.prevalences:
            idx = sample_batch(y, prev, self.batch_size)
            yield idx, prev

# ============================================================
# DyS (manual)
# ============================================================

def hellinger(p, q):
    p = p / (p.sum() + 1e-12)
    q = q / (q.sum() + 1e-12)
    return (1 / np.sqrt(2)) * np.sqrt(np.sum((np.sqrt(p) - np.sqrt(q)) ** 2))

def dys_quantifier(scores_batch, scores_train, y_train):
    pos_scores = scores_train[y_train == 1]
    neg_scores = scores_train[y_train == 0]

    h_batch, _ = np.histogram(scores_batch, bins=20, density=True)
    h_pos, _ = np.histogram(pos_scores, bins=20, density=True)
    h_neg, _ = np.histogram(neg_scores, bins=20, density=True)

    def mixture(p):
        return (1 - p) * h_neg + p * h_pos

    grid = np.linspace(0, 1, 1001)
    return float(min(grid, key=lambda p: hellinger(h_batch, mixture(p))))

# ============================================================
# LOAD MOSS REGRESSOR
# ============================================================

def load_moss_regressor(pkl_path):
    with open(pkl_path, "rb") as f:
        synthetic_dists = pickle.load(f)

    Xf, Y = [], []

    for key, curves in synthetic_dists.items():
        scores = np.vstack(curves)[:, 0]
        Xf.append(extract_features(scores))
        Y.append(key[0] if isinstance(key, tuple) else key)

    Xf = np.vstack(Xf)
    Y = np.array(Y)

    reg = RandomForestRegressor(
        n_estimators=300,
        random_state=SEED,
        n_jobs=-1
    )
    reg.fit(Xf, Y)

    return reg

# ============================================================
# LOAD DATASET
# ============================================================

def load_dataset(path):
    df = pd.read_csv(path)

    if "y" in df.columns:
        y = df["y"].values
        X = df.drop(columns=["y"]).values
    else:
        y = df.iloc[:, -1].values
        X = df.iloc[:, :-1].values

    if len(np.unique(y)) > 2:
        y = (y == np.max(y)).astype(int)

    return X, y

# ============================================================
# AVALIAÇÃO COMPLETA (COM TEMPO)
# ============================================================

def evaluate_dataset(ds_name, reg_moss, app):
    X, y = load_dataset(os.path.join(DATASETS_ROOT, ds_name))
    X = StandardScaler().fit_transform(X)

    Xtr, Xte, ytr, yte = train_test_split(
        X, y, test_size=0.5, stratify=y, random_state=SEED
    )

    clf = RandomForestClassifier(
        n_estimators=300,
        random_state=SEED,
        n_jobs=-1
    )
    clf.fit(Xtr, ytr)

    train_scores = clf.predict_proba(Xtr)[:, 1]

    dys = DyS(learner=clf, measure="topsoe")
    dyssyn = QuaDapt(
        quantifier=dys,
        merging_factors=[0.1, 0.3, 0.5, 0.7, 0.9],
        measure="topsoe"
    )
    dyssyn.fit(Xtr, ytr)

    rows = []

    for idx, _ in app.split(Xte, yte):
        Xb = Xte[idx]
        yb = yte[idx]

        scores = clf.predict_proba(Xb)[:, 1]
        prev_real = np.mean(yb)
        n = len(yb)

        # =========================
        # MOSS
        # =========================
        t0 = time.perf_counter()
        feats = extract_features(scores).reshape(1, -1)
        prev_moss = reg_moss.predict(feats)[0]
        t_moss = (time.perf_counter() - t0) / n

        # =========================
        # DyS
        # =========================
        t0 = time.perf_counter()
        prev_dys = dys_quantifier(scores, train_scores, ytr)
        t_dys = (time.perf_counter() - t0) / n

        # =========================
        # DySSyn
        # =========================
        t0 = time.perf_counter()
        prev_dyssyn = dyssyn.predict(Xb)[1]
        t_dyssyn = (time.perf_counter() - t0) / n

        rows.append({
            "arquivo_moss": MOSS_PKL,
            "dataset": ds_name,
            "prevalencia_real": prev_real,

            "prev_moss": prev_moss,
            "erro_moss": abs(prev_moss - prev_real),
            "tempo_moss_por_amostra": t_moss,

            "prev_dys": prev_dys,
            "erro_dys": abs(prev_dys - prev_real),
            "tempo_dys_por_amostra": t_dys,

            "prev_dyssyn": prev_dyssyn,
            "erro_dyssyn": abs(prev_dyssyn - prev_real),
            "tempo_dyssyn_por_amostra": t_dyssyn,

            "n_amostras": n,
            "var_scores": np.var(scores),
            "entropy": extract_features(scores)[-1]
        })

    return rows

# ============================================================
# MAIN
# ============================================================

if __name__ == "__main__":

    print("🔹 Carregando regressor MOSS...")
    reg_moss = load_moss_regressor(os.path.join(MOSS_FOLDER, MOSS_PKL))

    datasets = sorted([
        f for f in os.listdir(DATASETS_ROOT)
        if f.endswith(".csv")
    ])

    app = APP(
        batch_size=BATCH_SIZE,
        n_prev=N_PREV,
        repeats=REPEATS,
        seed=SEED
    )

    all_rows = []

    for ds in datasets:
        print(f"→ Dataset: {ds}")
        rows = evaluate_dataset(ds, reg_moss, app)
        all_rows.extend(rows)

    df = pd.DataFrame(all_rows)
    df.to_csv("tabela_final_MOSS_DyS_DySSyn.csv", index=False)

    print("\n✅ Experimento finalizado")
    print("📄 Arquivo salvo: tabela_final_MOSS_DyS_DySSyn.csv")

🔹 Carregando regressor MOSS...
→ Dataset: 1460_banana.csv
→ Dataset: 1462_banknote-authentication.csv


In [1]:
print("🔹 Carregando regressor MOSS...")

🔹 Carregando regressor MOSS...


In [ ]:
plt.figure(figsize=(12,6))

# MOSS (um por arquivo)
for moss in arquivos_moss:
    sub = df[df["arquivo_moss"] == moss]
    plt.scatter(
        sub["prevalencia_real"],
        sub["erro_moss"],
        label=f"MOSS – {moss}",
        alpha=0.5
    )

# DyS
plt.scatter(
    df["prevalencia_real"],
    df["erro_dys"],
    label="DyS",
    color="black",
    alpha=0.6
)

# DySSyn
plt.scatter(
    df["prevalencia_real"],
    df["erro_dyssyn"],
    label="DySSyn",
    color="red",
    alpha=0.6
)

plt.xlabel("Prevalência real")
plt.ylabel("Erro absoluto")
plt.title("Erro × Prevalência real — MOSS vs DyS vs DySSyn")
plt.legend()
plt.tight_layout()
plt.show()

dfs = []

# MOSS
for moss in arquivos_moss:
    dfs.append(pd.DataFrame({
        "Método": f"MOSS – {moss}",
        "Erro": df[df["arquivo_moss"] == moss]["erro_moss"]
    }))

# DyS
dfs.append(pd.DataFrame({
    "Método": "DyS",
    "Erro": df["erro_dys"]
}))

# DySSyn
dfs.append(pd.DataFrame({
    "Método": "DySSyn",
    "Erro": df["erro_dyssyn"]
}))

df_melt = pd.concat(dfs, ignore_index=True)

# ordenar do melhor para o pior (menor mediana)
ordem_metodos = (
    df_melt
    .groupby("Método")["Erro"]
    .median()
    .sort_values()
    .index
    .tolist()
)

plt.figure(figsize=(12,6))

sns.boxplot(
    data=df_melt,
    x="Método",
    y="Erro",
    order=ordem_metodos
)

plt.xticks(rotation=45)
plt.ylim(0, 0.4)
plt.title(
    "Distribuição dos erros — MOSS vs DyS vs DySSyn\n"
    "(ordenado do melhor para o pior)"
)
plt.tight_layout()
plt.show()

In [ ]:
import pandas as pd
import plotly.express as px

# carregar dados
df = pd.read_csv("tabela_final_MOSS.csv")

# escolher qual MOSS usar
MOSS_ESCOLHIDO = "moss_0_100.pkl"

# DyS
df_dys = (
    df
    .groupby("dataset", as_index=False)
    .agg({"erro_dys": "mean"})
)
df_dys["Método"] = "DyS"
df_dys = df_dys.rename(columns={"erro_dys": "Erro"})

# DySSyn
df_dyssyn = (
    df
    .groupby("dataset", as_index=False)
    .agg({"erro_dyssyn": "mean"})
)
df_dyssyn["Método"] = "DySSyn"
df_dyssyn = df_dyssyn.rename(columns={"erro_dyssyn": "Erro"})

# MOSS
df_moss = (
    df[df["arquivo_moss"] == MOSS_ESCOLHIDO]
    .groupby("dataset", as_index=False)
    .agg({"erro_moss": "mean"})
)
df_moss["Método"] = "MOSS 0–100"
df_moss = df_moss.rename(columns={"erro_moss": "Erro"})

# juntar tudo
df_long = pd.concat(
    [df_moss, df_dys, df_dyssyn],
    ignore_index=True
)

# plot
fig = px.line(
    df_long,
    x="dataset",
    y="Erro",
    color="Método",
    markers=True,
    title="Erro médio por dataset — MOSS 0–100 vs DyS vs DySSyn",
)

fig.update_layout(
    xaxis_title="Dataset",
    yaxis_title="Erro absoluto médio",
    legend_title="Método",
    hovermode="x unified"
)

fig.update_xaxes(tickangle=45)

# 💾 salvar em HTML
fig.write_html("erro_medio_por_dataset_MOSS_DyS_DySSyn.html")

# opcional: mostrar na tela
fig.show()

In [ ]:
import pandas as pd

# carregar dados
df = pd.read_csv("tabela_final_MOSS.csv")

MOSS_ESCOLHIDO = "moss_0_100.pkl"

# erro médio por dataset
df_agg = (
    df
    .groupby("dataset", as_index=False)
    .agg({
        "erro_dys": "mean",
        "erro_dyssyn": "mean"
    })
)

df_moss = (
    df[df["arquivo_moss"] == MOSS_ESCOLHIDO]
    .groupby("dataset", as_index=False)
    .agg({"erro_moss": "mean"})
)

# juntar tudo numa tabela só
df_cmp = (
    df_agg
    .merge(df_moss, on="dataset", how="inner")
)

# determinar vencedor por dataset
def vencedor(row):
    erros = {
        "MOSS 0–100": row["erro_moss"],
        "DyS": row["erro_dys"],
        "DySSyn": row["erro_dyssyn"]
    }
    return min(erros, key=erros.get)

df_cmp["Vencedor"] = df_cmp.apply(vencedor, axis=1)

# flag: MOSS melhor que os dois
df_cmp["MOSS_vence_ambos"] = (
    (df_cmp["erro_moss"] < df_cmp["erro_dys"]) &
    (df_cmp["erro_moss"] < df_cmp["erro_dyssyn"])
)

# resumo de vitórias
resumo_vitorias = (
    df_cmp["Vencedor"]
    .value_counts()
    .rename("N_datasets")
    .reset_index()
    .rename(columns={"index": "Método"})
)

# métrica chave
n_moss_vence_ambos = df_cmp["MOSS_vence_ambos"].sum()
total_datasets = df_cmp.shape[0]

print("=== VITÓRIAS POR MÉTODO ===")
print(resumo_vitorias)

print("\n=== MOSS vence DyS e DySSyn simultaneamente ===")
print(f"{n_moss_vence_ambos} / {total_datasets} datasets "
      f"({100 * n_moss_vence_ambos / total_datasets:.1f}%)")


dar uma olhada para os 15 datasets onde ele foi melhor, fazer por 30 repetições, agregar mais features que permita descrever mais picos locais e globais. extratores de características